# **🤖 Data Analytics AI Agent 🤖** 

This AI-powered Analytics Agent, will have for goal investigate how AI assistants such as ChatGPT are used by students at different academic levels use AI tools like ChatGPT for tasks such as coding, writing, studying, and brainstorming. Designed for learning, EDA, and ML experimentation.


# This AI Agent should be able to:

1. **🧹 Clean:** Agent will clean messy data, handle missing values, and structure the datasets for analysis.
2. **🔢 Process/Analyze:** Users can ask questions in plain English (e.g., "What was the average number of students using AI assistants?") and agent will be able to produce an answer to the query.
3. **📊 Visualize:** Agent can automatically create charts, graphs, to illustrate data trend.
4. **📋 Provide a conclusion:** Agent will summarize findings, create reports, and recommend actions.

# About the Dataset 🗄️

**AI Assistant Usage in Student Life**

This dataset simulates 10,000 sessions of students interacting with an AI assistant (like ChatGPT or similar tools) for various academic tasks. Each row represents a single session, capturing the student’s level, discipline, type of task, session length, AI effectiveness, satisfaction rating, and whether they reused the AI tool later.

Link to download:
https://www.kaggle.com/datasets/ayeshasal89/ai-assistant-usage-in-student-life-synthetic?resource=download


**Variables**
- SessionID--Unique session identifier
- StudentLevel--Academic level: High School, Undergraduate, Graduate
- Discipline--Student’s field of study (e.g., CS, Psychology, etc.)
- SessionDate--Date of the session
- SessionLengthMin--Length of AI interaction in minutes
- TotalPrompts--Number of prompts/messages used
- TaskType--Nature of the task (e.g., Coding, Writing, Research)
- AI_AssistanceLevel--1–5 scale on how helpful the AI was perceived to be
- FinalOutcome--What the student achieved: Assignment Completed, Idea Drafted, etc.
- UsedAgain--Whether the student returned to use the assistant again
- SatisfactionRating--1–5 rating of overall satisfaction with the session


# Let's get to business 🤓

In [6]:
# Install dependencies
!pip install openai pandas matplotlib

Defaulting to user installation because normal site-packages is not writeable


In [75]:
!pip install tabulate

147281.54s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


Defaulting to user installation because normal site-packages is not writeable
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [7]:
%pip install matplotlib

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [40]:
# Core imports
import pandas as pd
import os
from openai import OpenAI
import matplotlib.pyplot as plt
from typing import TypedDict, List, Callable, Dict, Any, Optional, Type, Union
from pydantic import BaseModel
import inspect
import sys
from io import StringIO

# Initialize OpenAI client
openai_client = OpenAI()

def generate(
    prompt: str,
    temperature: float = 0,
    response_format: Optional[Type[BaseModel]] = None,
    model: str = "gpt-4o-mini"
) -> Union[str, BaseModel]:
    """
    🎨 Generate text using OpenAI's API with optional structured output

    Args:
        prompt: The input prompt for generation
        temperature: Sampling temperature (0-2), default 0
        response_format: Optional Pydantic model class for structured output
        model: The model to use, default "gpt-4o-mini"

    Returns:
        Either a string (regular generation) or a Pydantic model instance (structured output)
    """
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt}
    ]

    if response_format is not None:
        # Use structured output with Pydantic model
        response = openai_client.beta.chat.completions.parse(
            model=model,
            messages=messages,
            temperature=temperature,
            response_format=response_format
        )
        return response.choices[0].message.parsed
    else:
        # Regular text generation
        response = openai_client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=temperature
        )
        return response.choices[0].message.content.strip()

print("✅ Core utilities loaded")

✅ Core utilities loaded


# Load the dataset  🗄️

In [30]:
try:
    df = pd.read_csv("ai_assistant_usage_student_life.csv")
    print("File loaded successfully!")
    df.columns = df.columns.str.replace(' ', '_').str.replace('.', '').str.lower()
    print(f"✅ Loaded {len(df)} Dataset with {len(df.columns)} columns")
    print(df.head(5))
except FileNotFoundError:
    print("The system can't find the file. Check the 'Data' tab on the right.")

File loaded successfully!
✅ Loaded 10000 Dataset with 11 columns
      sessionid   studentlevel        discipline sessiondate  \
0  SESSION00001  Undergraduate  Computer Science  2024-11-03   
1  SESSION00002  Undergraduate        Psychology  2024-08-25   
2  SESSION00003  Undergraduate          Business  2025-01-12   
3  SESSION00004  Undergraduate  Computer Science  2025-05-06   
4  SESSION00005  Undergraduate        Psychology  2025-03-18   

   sessionlengthmin  totalprompts  tasktype  ai_assistancelevel  \
0             31.20            11  Studying                   2   
1             13.09             6  Studying                   3   
2             19.22             5    Coding                   3   
3              3.70             1    Coding                   3   
4             28.12             9   Writing                   3   

           finaloutcome  usedagain  satisfactionrating  
0  Assignment Completed       True                 1.0  
1  Assignment Completed       Tru

# **🧹🧹Cleaning tools!🧹🧹**

In [41]:
from difflib import get_close_matches

# Tool 1: Academic Level Standardizer
def standardize_student_levels(df: pd.DataFrame):
    """
    Ensures StudentLevel is strictly: 'High School', 'Undergraduate', or 'Graduate'.
    Fixes common synthetic issues like 'undergrad' or 'highschool'.
    """
    valid_levels = ['High School', 'Undergraduate', 'Graduate']
    column = 'studentlevel'
    
    def match_level(val):
        if pd.isna(val): return val
        matches = get_close_matches(val.title(), valid_levels, n=1, cutoff=0.6)
        return matches[0] if matches else "Other"

    df[column] = df[column].apply(match_level)
    return f"Standardized {column} to primary academic tiers."

# Tool 2: Session Logic Validator
def validate_session_metrics(df: pd.DataFrame):
    """
    Identifies 'Impossible Sessions'. 
    Example: TotalPrompts > 0 but SessionLengthMin is 0, or vice versa.
    """
    # Flagging rows where AI was used but no time was recorded
    mask = (df['totalprompts'] > 0) & (df['sessionlengthmin'] <= 0)
    invalid_count = mask.sum()
    
    # Impute 1 minute for sessions with prompts but 0 time
    df.loc[mask, 'sessionlengthmin'] = 1
    
    return f"Fixed {invalid_count} sessions with prompts but zero duration."

# Tool 3: Discipline Grouper
def group_disciplines(df: pd.DataFrame, top_n: int = 10):
    """
    In this dataset, 'Discipline' can be very fragmented (e.g., 'CS', 'CompSci').
    This tool groups infrequent disciplines into 'Other' to prevent overfitting.
    """
    counts = df['discipline'].value_counts()
    keep = counts.nlargest(top_n).index
    df['discipline'] = df['discipline'].where(df['discipline'].isin(keep), 'Other')
    
    return f"Consolidated disciplines. Top {top_n} retained, others moved to 'Other'."

# Tool 4: Satisfaction Rating Clipper
def sanitize_ratings(df: pd.DataFrame):
    """
    SatisfactionRating and AI_AssistanceLevel should be 1-5.
    This tool clips any synthetic noise outside this range.
    """
    for col in ['satisfactionrating', 'ai_assistancelevel']:
        df[col] = df[col].clip(1, 5).round(0).astype(int)
        
    return "Clipped and rounded ratings to strict 1-5 integer scale."


"""
"Check for Fairness" Tool
As this is educational data, the agent should also check for class imbalance. If "Graduate" students only make up 
1% of your data, the agent's analytics will be biased toward "Undergraduates."
"""
def check_representation(df: pd.DataFrame, column: str = 'studentlevel'):
    """Reports the percentage distribution of a category to alert for bias."""
    distribution = df[column].value_counts(normalize=True) * 100
    report = distribution.to_dict()
    return f"Representation Check for {column}: {report}"

# **🔢🔢 Processing and Analyzing Tools 🔢🔢**

In [76]:

def calculate_column_stats(df: pd.DataFrame, column: str, group_by: str = None):
    """
    Calculates statistics and returns a formatted Markdown table for Gradio.
    """
    # 1. Validation
    if column not in df.columns:
        return f"⚠️ Error: Column '{column}' not found."
    
    if not pd.api.types.is_numeric_dtype(df[column]):
        return f"⚠️ Error: '{column}' is not numeric and cannot be aggregated."

    stats_list = ['mean', 'median', 'max', 'min', 'count', 'std']

    if group_by and group_by in df.columns:
        # 2. Perform grouped aggregation
        report = df.groupby(group_by)[column].agg(stats_list).round(2)
        # 3. Use .to_markdown() for beautiful Gradio tables
        return report.to_markdown()
    else:
        # 4. Perform global aggregation
        report = df[column].agg(stats_list).round(2)
        # Convert Series to DataFrame so it renders as a table
        return report.to_frame().T.to_markdown(index=False)
        
def check_correlation(df: pd.DataFrame, col1: str, col2: str):
    """Calculates the Pearson correlation between two numeric columns."""
    correlation = df[col1].corr(df[col2])
    
    if correlation > 0.7:
        strength = "strong positive"
    elif correlation < -0.7:
        strength = "strong negative"
    else:
        strength = "weak"
        
    return f"The correlation between {col1} and {col2} is {correlation:.2f}, which is a {strength} relationship."

# **📊📊 Visualizing Tools 📊📊**

In [22]:
#need it for pretty charts
#pip install seaborn

In [65]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set a clean visual style for all plots
sns.set_theme(style="whitegrid")

def plot_histogram_plt(df: pd.DataFrame, column: str):
    """Shows the distribution/frequency of a numeric column."""
    plt.figure(figsize=(10, 6))
    sns.histplot(df[column], kde=True, color='skyblue')
    plt.title(f'Distribution of {column}')
    plt.xlabel(column)
    plt.ylabel('frequency')
    plt.show()
    return f"Histogram of {column} rendered."

def plot_bar_chart_plt(df: pd.DataFrame, x_col: str, y_col: str, operation: str = "mean"):
    """Compares a metric (y) across categories (x)."""
    fig = plt.figure(figsize=(10, 6))
    
    # Seaborn's barplot can handle the 'mean' calculation automatically
    # 'estimator' can be np.mean, np.median, np.sum, or len (for count)
    import numpy as np
    estimator_map = {"mean": np.mean, "median": np.median, "sum": np.sum, "count": len}
    
    sns.barplot(
    data=df, 
    x=x_col, 
    y=y_col, 
    estimator=estimator_map.get(operation, np.mean), 
    hue=x_col,      
    legend=False,   
    palette="viridis"
)
    plt.title(f'{operation.title()} {y_col} by {x_col}')
    plt.xticks(rotation=45)
    
    return fig

def plot_scatter_plt(df: pd.DataFrame, x_col: str, y_col: str, color_by: str = None):
    """Shows the relationship between two numbers."""
    fig = plt.figure(figsize=(10, 6))
    sns.scatterplot(data=df, x=x_col, y=y_col, hue=color_by, palette="magma", alpha=0.7)
    
    # Add a trend line for better analysis
    sns.regplot(data=df, x=x_col, y=y_col, scatter=False, color='red')
    
    plt.title(f'Relationship: {x_col} vs {y_col}')
    
    return fig

def plot_box_plot_plt(df: pd.DataFrame, y_col: str, x_col: str = None):
    """Shows variance and outliers."""
    fig = plt.figure(figsize=(10, 6))
    sns.boxplot(data=df, x=x_col, y=y_col, palette="Set2")
    
    plt.title(f'Box Plot of {y_col}' + (f' across {x_col}' if x_col else ""))
    
    return fig
    

# **📋 Providing a conclusion 📋**

In [53]:

def generate_summary_report(df: pd.DataFrame, stats_results: dict, insight_goal: str):
    """
    Combines raw data stats with a high-level goal to produce a 
    structured report. This tool helps the agent 'focus' its conclusion.
    """
    report_template = f"""
    --- DATA ANALYTICS REPORT ---
    GOAL: {insight_goal}
    TOTAL SESSIONS ANALYZED: {len(df)}
    
    KEY METRICS:
    {stats_results}
    
    CRITICAL FINDINGS:
    - The average AI usage per session is {df['sessionlengthmin'].mean():.2f} minutes.
    - Highest adoption is seen in the {df['studentlevel'].mode()[0]} demographic.
    - User satisfaction correlates with {df[['satisfactionrating', 'totalprompts']].corr().iloc[0,1]:.2f} prompt frequency.
    """
    return report_template

def recommend_actions(df: pd.DataFrame):
    """
    Uses data thresholds to trigger specific recommendations.
    This moves the agent from 'What happened' to 'What should we do'.
    """
    recommendations = []
    
    # Logic-based recommendations
    avg_satisfaction = df['satisfactionrating'].mean()
    if avg_satisfaction < 3.5:
        recommendations.append("LOW SATISFACTION: Investigate 'TaskType' to see where AI is failing students.")
    
    undergrad_usage = df[df['studentlevel'] == 'Undergraduate']['totalprompts'].mean()
    grad_usage = df[df['studentlevel'] == 'graduate']['totalprompts'].mean()
    
    if grad_usage > undergrad_usage * 1.5:
        recommendations.append("ADOPTION GAP: Undergraduates are using AI significantly less than Graduates. Consider introductory workshops.")

    return recommendations if recommendations else ["Data looks stable. Maintain current support levels."]
    

# ***THE* Demo**

In [26]:
pip install gradio

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.0/43.0 MB 164.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.2/625.2 kB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 195.4 MB/s eta 0:00:00
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn

In [85]:
import gradio as gr
import matplotlib.pyplot as plt

# 1. PRE-PROCESS: Run cleaning tools ONCE before launching the UI
print("Cleaning data...")
df_cleaned = df.copy()
standardize_student_levels(df_cleaned)
validate_session_metrics(df_cleaned)
sanitize_ratings(df_cleaned)

# 2. THE MASTER LOGIC FUNCTION
#Agent's thinking structure
from pydantic import BaseModel, Field
from typing import Literal, Optional


class AgentDecision(BaseModel):
    tool_name: Literal["clean", "stats", "plot", "report"] = Field(description="The tool to use")
    column_name: str = Field(description="The numeric column to analyze")
    group_by_column: Optional[str] = Field(None, description="The category to group by (e.g., 'StudentLevel') if comparing groups")
    justification: str = Field(description="Why this tool and grouping was chosen")

# Initializing the memory list
history_buffer = []


def master_agent_interface(query):
    global history_buffer
    plt.close('all')
    
    # Step 1. Ask the LLM to decide what to do
    past_context = "\n\n".join(history_buffer)
    system_context = f"""
    You are a Data Science Router. 
    Columns available: {df_cleaned.columns.tolist()}
    
    RULES:
    1. If the user asks to 'compare' or asks 'which level/group', you MUST set group_by_column to 'StudentLevel'.
    2. 'TotalPrompts' and 'SatisfactionRating' are numeric. 'StudentLevel' is categorical.

    
    RECENT MEMORY (Previous 4 findings):
    {past_context if past_context else "No previous history."}
    
    User Query: {query}
    """
    
    # Using existing generate function with the Pydantic model
    decision = generate(system_context, response_format=AgentDecision, model="gpt-4o-mini")
    
    # Step 2. Execute the tool based on the LLM's decision
    fig = None
    conclusion = ""
    stats_output = ""
    
    if decision.tool_name == "plot":
        # The LLM picked 'plot' and identified the column!
        fig = plot_bar_chart_plt(df_cleaned, x_col='studentlevel', y_col=decision.column_name)
        stats_output = calculate_column_stats(
            df_cleaned, 
            column=decision.column_name, 
            group_by=decision.group_by_column
        )
        conclusion = f"### Statistics for {decision.column_name} (Grouped by {decision.group_by_column})\n{stats_output}"

    elif decision.tool_name == "stats":
        stats_output = calculate_column_stats(
            df_cleaned, 
            column=decision.column_name, 
            group_by=decision.group_by_column
        )
        conclusion = f"### Statistics for {decision.column_name} (Grouped by {decision.group_by_column})\n{stats_output}"

    elif decision.tool_name == "report":
        conclusion = generate_summary_report(df_cleaned, {}, query)
        
    
    # Step 3. Ask the LLM to summarize findings
    summary_prompt = f"""
    The user asked: {query}
    Here are the results of the data analysis:
    {stats_output}
    
    Provide a brief, 2-sentence executive summary. 
    Format it exactly like this: 
    "The level with the highest {decision.column_name} is [Level] because [Observation from data]."
    """
    
    ai_summary = generate(summary_prompt, model="gpt-4o-mini")
    
    # 4. Combine everything for the UI
    final_output = f"### 📊 Analysis Findings\n\n{stats_output}\n\n**Conclusion:** {ai_summary}"
    
    # Safety: ensure a figure exists for Gradio
    if fig is None:
        fig = plt.figure(figsize=(1,1))
        #plt.axis('off')
    #Add newest result to memory
    history_buffer.append(f"**Q:** {query} | **A:** {ai_summary}")
    if len(history_buffer) > 4:
        history_buffer.pop(0)
    # Create a formatted string to show in the UI
    history_md = "### 📜 Recent History\n" + "\n\n".join(history_buffer)

        
    return fig, final_output, history_md

# 3. GRADIO LAYOUT
with gr.Blocks() as demo:
    gr.Markdown("# 🤖 Data Analytics AI Agent 🤖 ")
    
    with gr.Tab("Analysis Dashboard"):
        with gr.Row():
            with gr.Column(scale=1):
                user_input = gr.Textbox(label="Ask your Agent", placeholder="e.g., Show me a plot of prompts by level")
                submit_btn = gr.Button("Execute", variant="primary")
                history_display = gr.Markdown("History will appear here after the first query.")
            
            with gr.Column(scale=2):
                plot_output = gr.Plot(label="Visualization Tool Output")
                text_output = gr.Markdown(label="Agent Analysis")

    with gr.Tab("Data Health"):
        gr.Markdown("### Current Cleaned Data Preview")
        gr.Dataframe(df_cleaned.head(10))

    submit_btn.click(
        fn=master_agent_interface, 
        inputs=user_input, 
        outputs=[plot_output, text_output, history_display]
    )

demo.launch(share=True, inline=False)

Cleaning data...
* Running on local URL:  http://127.0.0.1:7889
* Running on public URL: https://a26082a20e07530f60.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


### 🚀 Next Steps for me:
1. Github repository
2. Pretty presentation
5. Better demo?
6. Suggestions?